# eicu-crd: SOFA validation against RICU

Validate the six SOFA components and total SOFA score from WeavEHR against the RICU reference implementation over the first 7 ICU days.


In [ ]:
import os
import subprocess
from pathlib import Path

from weavehr_yaib import (
    build_and_write_yaib_wide_for_dataset,
    compare_weavehr_wide_to_ricu_for_dataset,
    concept_root_from_output,
    resolve_weavehr_workspace,
)

DATASET = "eicu-crd"
MAX_HOURS = 7 * 24

# WeavEHR dataset version to use (e.g. "1.5.0"). None resolves it from the
# WeavEHR project and fails if several versions are present.
DATASET_VERSION = None

WEAVEHR_OUTPUT = (
    Path.home()
    / "output"
    / "WeavEHR.example"
    / "eicu-crd.sofa"
    / "project"
)

RICU_DYNAMIC_VARS = [
    "sofa_resp",
    "sofa_coag",
    "sofa_liver",
    "sofa_cardio",
    "sofa_cns",
    "sofa_renal",
    "sofa",
]

CONCEPT_MAPPING = {
    "sofa_resp": "sofa_respiratory",
    "sofa_coag": "sofa_coagulation",
    "sofa_liver": "sofa_liver",
    "sofa_cardio": "sofa_cardiovascular",
    "sofa_cns": "sofa_cns",
    "sofa_renal": "sofa_renal",
    "sofa": "sofa",
}

WEAVEHR_OUTPUT = WEAVEHR_OUTPUT.expanduser().resolve()
WORKSPACE = resolve_weavehr_workspace(WEAVEHR_OUTPUT)
CONCEPT_ROOT = concept_root_from_output(WEAVEHR_OUTPUT)

DATASET_OUTPUT = WORKSPACE / "yaib" / "sofa"
DATASET_OUTPUT.mkdir(parents=True, exist_ok=True)

STAYS_PATH = None

RICU_CONCEPT_DICT = (
    Path.home()
    / "workspace"
    / "ricu"
    / "inst"
    / "extdata"
    / "config"
    / "concept-dict.json"
)


In [ ]:
# This validation intentionally exports only the SOFA variables.
RICU_DYNAMIC_VARS


In [ ]:
# 1) Export the matching SOFA variables from RICU.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Could not locate the weavehr-yaib repository root.")

env = os.environ.copy()
env["RICU_OUT_DIR"] = str(DATASET_OUTPUT)
# Declare the version of the RICU source data in RICU_DATA_PATH if known. Without
# it the reference version is only assumed (RICU config URL or manual default).
# env["RICU_SOURCE_VERSION"] = "..."
env["RICU_DYNAMIC_VARS"] = ",".join(RICU_DYNAMIC_VARS)

if "RICU_DATA_PATH" not in env:
    raise ValueError("Set RICU_DATA_PATH before running the RICU export.")

env["RICU_SRC_LOAD"] = "eicu"
env.pop("RICU_CONFIG_PATH", None)
env["R_ENVIRON_USER"] = "/dev/null"

subprocess.run(
    [
        "Rscript",
        "--no-environ",
        "scripts/datasets/export_ricu_eicu-crd.R",
    ],
    cwd=REPO_ROOT,
    check=True,
    env=env,
)


In [ ]:
# 2) Build the WeavEHR SOFA wide representation for the first 7 days.
weavehr_7d = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    version="1.0.0",
    dynamic_vars=RICU_DYNAMIC_VARS,
    concept_mapping=CONCEPT_MAPPING,
    aggregation_mode="ricu",
)

print("WeavEHR SOFA export written to:", weavehr_7d.output_path)


In [ ]:
# 3) Compare WeavEHR with RICU.
comparison = compare_weavehr_wide_to_ricu_for_dataset(
    dataset=DATASET,
    dataset_version=DATASET_VERSION,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    weavehr_wide_path=weavehr_7d.output_path,
    dynamic_vars=RICU_DYNAMIC_VARS,
)

# Display aggregate/statistical reports only.
safe_reports = {
    "table_summary": comparison.table_summary,
    "coverage": comparison.coverage,
    "missingness": comparison.missingness,
    "value_diff": comparison.value_diff,
    "reproduction_accuracy": comparison.reproduction_accuracy,
}

safe_reports
